# Train the ticket triage models on Google Colab

This notebook runs the whole pipeline on Colab's free GPU:

1. Download and clean the dataset
2. Train the baselines (majority class, TF-IDF + logistic regression)
3. Fine-tune DistilBERT for **team (queue)**, **priority** and **ITIL type**
4. Write the results table and charts into the README
5. *(Optional)* upload the models to the Hugging Face Hub for the live demo

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

In [ ]:
# Check that a GPU is attached (should print something like "Tesla T4")
!nvidia-smi -L

## 1. Get the code
Replace `rushitpatel100-oss` with your GitHub username.

In [ ]:
REPO_URL = "https://github.com/rushitpatel100-oss/ticket-triage-ai"

!git clone -q {REPO_URL}
%cd ticket-triage-ai
!pip install -q -r requirements.txt

## 2. Prepare the data (about 1 minute)

In [ ]:
!python -m src.data

## 3. Baselines (a few minutes)
Always build a simple baseline first, so you can show the transformer is actually worth it.

In [ ]:
!python -m src.baseline

## 4. Fine-tune DistilBERT (roughly 20-40 minutes for all three tasks on a T4)

Want a quick trial run first? Add `--max-train-samples 3000` to the command below.
You can also train one task at a time, e.g. `--task queue`.

In [ ]:
!python -m src.train

## 5. Results table and charts
This prints the table and writes it, with the charts, into `README.md`.

In [ ]:
!python -m src.report

from IPython.display import Image, display
display(Image("results/model_comparison.png"))

Look at the model's most confident mistakes. Are the labels wrong, or is the model? Write 3 or 4 findings in the README.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)
pd.read_csv("results/errors_queue_transformer.csv").head(10)

## 6. (Optional) Upload the models to the Hugging Face Hub

Needed for the live demo on Hugging Face Spaces. Create a **write** token at
https://huggingface.co/settings/tokens and paste it into the login box. Never put the token in your code.

In [ ]:
from huggingface_hub import HfApi, notebook_login

notebook_login()

In [ ]:
HF_USER = "your-hf-username"

api = HfApi()
for task in ["queue", "priority", "type"]:
    repo_id = f"{HF_USER}/ticket-triage-{task}"
    api.create_repo(repo_id, exist_ok=True)
    api.upload_folder(folder_path=f"models/{task}", repo_id=repo_id)
    print("Uploaded", f"https://huggingface.co/{repo_id}")

## 7. Download the results

This downloads `results/` and the updated `README.md`. Unzip it into your local project folder
(overwrite when asked), then commit and push to GitHub.

In [ ]:
!zip -q -r results_and_readme.zip results README.md
from google.colab import files
files.download("results_and_readme.zip")